# Grokking dynamics: Stage 1 on a Colab GPU

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. When an upload button appears under cell 2, choose `grokking-dynamics.zip`.
4. Keep this tab open. The last cell downloads `stage1_results.zip`.

If Colab asks you to *restart the session* after the install cell, click Restart, then **Run all** again (the upload is skipped the second time).

The notebook trains 10 seeds with weight decay 1, one control with weight decay 0, and a from-scratch re-run of seed 0 to check byte-identical reproducibility. It runs the tests before and after training and draws the figures.

In [ ]:
# 1. Check that a GPU is attached
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again."
print(torch.__version__, "CUDA", torch.version.cuda, "-", torch.cuda.get_device_name(0))

In [ ]:
# 2. Upload and unpack the code (choose grokking-dynamics.zip)
import os, zipfile
os.chdir("/content")
if not os.path.isdir("/content/grokking-dynamics"):
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
os.chdir("/content/grokking-dynamics")
print(sorted(os.listdir()))

In [ ]:
# 3. Install dependencies and record the environment
!pip install -q "transformer_lens>=2.0,<4" pytest pyyaml
!mkdir -p logs results/01_baseline
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch, numpy, importlib.metadata as m; print('torch', torch.__version__, 'cuda', torch.version.cuda); print('numpy', numpy.__version__); print('transformer_lens', m.version('transformer_lens'))") | tee results/01_baseline/ENVIRONMENT.txt

In [ ]:
# 4. Tests before training (includes a GPU determinism check)
!python -m pytest tests -q -p no:warnings 2>&1 | tee logs/pytest_before.log

In [ ]:
# 5. Train: 10 seeds at weight decay 1, then the weight-decay-0 control. Prints progress every 1000 steps.
!python experiments/01_baseline.py --config configs/baseline.yaml --seeds 0-9 --jobs 2 2>&1 | tee logs/train_baseline.log
!python experiments/01_baseline.py --config configs/baseline_wd0.yaml --seed 0 2>&1 | tee logs/train_wd0.log

In [ ]:
# 6. Reproducibility: re-train seed 0 from scratch and byte-compare its CSV (expect MATCH)
!python experiments/regenerate_checkpoints.py --config configs/baseline.yaml --seed 0 2>&1 | tee logs/repro_seed0.log

In [ ]:
# 7. Tests after training (now also ports trained checkpoints to TransformerLens)
!python -m pytest tests -q -p no:warnings 2>&1 | tee logs/pytest_after.log

In [ ]:
# 8. Figures and milestone table
!python figures/make_all.py 2>&1 | tee logs/make_all.log
from IPython.display import Image, display
display(Image("figures/fig01_accuracy_across_seeds.png"))
display(Image("figures/fig02_weight_decay_control.png"))

In [ ]:
# 9. Package results (CSVs, figures, logs, seed-0 checkpoints) and download
!rm -f stage1_results.zip && zip -q -r stage1_results.zip results figures/*.png logs checkpoints/01_baseline/baseline_seed0
!ls -lh stage1_results.zip
from google.colab import files
files.download("stage1_results.zip")